# CS781 Phase 2 – Legal NER: fetch, clean, audit, tokenise

**Before running**
1. On the Hugging Face page of `Exploration-Lab/CS-781-Capstone`: log in and **accept the conditions** (repo is gated).
2. Create a HF access token (read) and add it in Kaggle: *Add-ons → Secrets → `HF_TOKEN`*.
3. Notebook settings: **Internet ON**. No GPU needed.
4. Add your **tokenizer dataset** as input.

**What it does** (only Phase 2 files are downloaded)
- Cleaning is **length-preserving** (every fix swaps one character for one character), so all gold offsets and all test-set predictions stay valid with no remapping.
- Gold spans are validated against the text (end convention, wrong offsets, bad labels, duplicates, overlaps) and repaired where it is safe.
- Duplicate documents and train/validation/test leakage are reported (and duplicates removed from train folds).
- Texts are tokenised with your tokenizer, aligned to BIO tags, and an **oracle check** proves the boundaries survive tokenisation.

**Output** (`/kaggle/working/ner_data_clean/`): cleaned `Train/` and `Test/` JSON in the original format (drop-in for the NER notebook), `labels.txt`, `tokenized_ner.pkl`, `audit_report.json`.

In [ ]:
import os, sys, glob, json, re, time, hashlib, pickle, unicodedata
from collections import Counter
import numpy as np

# ======================= SETTINGS =======================
HF_REPO = "Exploration-Lab/CS-781-Capstone"
RAW_DIR = "/kaggle/working/raw_phase2"
OUT_DIR = "/kaggle/working/ner_data_clean"
TOKENIZER_DIR = None          # None = search /kaggle/input
LOCAL_DIR = None              # set to a folder that already holds the files to skip the download

# --- cleaning (all length-preserving) ---
CLEAN_TEXT = True
EXTRA_SPACE_CHARS = "​⁠﻿"   # zero-width chars that act as separators -> space
NORMALISE_PUNCT = False                     # smart quotes / dashes -> ascii (off: tokenizer saw the originals in Phase 1)

# --- gold validation ---
REPAIR_WINDOW = 60            # search +-N chars for a misplaced entity text
DEDUP_TRAIN = True            # drop duplicate docs inside a fold, and fold2 docs that also occur in fold1

# --- tokenisation (keep identical to the NER notebook) ---
ADD_EOS_PREFIX = True
MAX_LEN = 512
OVERLAP = 128
# [[END OF SETTINGS]]

for name in ["HF_REPO", "RAW_DIR", "OUT_DIR", "TOKENIZER_DIR", "LOCAL_DIR"]:
    v = os.environ.get("PREP_" + name)
    if v:
        globals()[name] = v
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
REPORT = {}

## 1. Download the Phase 2 files

In [ ]:
WANTED = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "fold3-text.json", "labels.txt"]

def get_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as err:
        print("no HF_TOKEN secret found:", err)
        return None

LOCAL_FILES = {}
if LOCAL_DIR is not None:
    for name in WANTED:
        found = glob.glob(os.path.join(LOCAL_DIR, "**", name), recursive=True)
        LOCAL_FILES[name] = found[0]
else:
    from huggingface_hub import list_repo_files, hf_hub_download
    token = get_token()
    try:
        repo_files = list_repo_files(HF_REPO, repo_type="dataset", token=token)
    except Exception as err:
        raise RuntimeError("Cannot list the repo. Check: Internet ON, HF_TOKEN secret set, conditions accepted on the dataset page.\n" + str(err))
    print("repo has", len(repo_files), "files")
    chosen = {}
    for path in repo_files:
        base = os.path.basename(path)
        low = path.lower().replace("-", "_")
        if base in WANTED and ("lner" in low or "phase_2" in low or "phase2" in low or "l_ner" in low):
            chosen[base] = path
    for name in WANTED:
        if name not in chosen:
            print("files that look related:", [p for p in repo_files if "fold" in p.lower() or "ner" in p.lower()][:30])
            raise FileNotFoundError(name + " not found in the repo listing")
    for name in WANTED:
        local = hf_hub_download(HF_REPO, filename=chosen[name], repo_type="dataset", token=token, local_dir=RAW_DIR)
        LOCAL_FILES[name] = local
        print("downloaded", chosen[name], f"({os.path.getsize(local) / 1e6:.1f} MB)")
print(LOCAL_FILES)

## 2. Load raw data

In [ ]:
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_texts(path):
    raw = read_json(path)
    texts = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            if isinstance(value, dict):
                value = value.get("text", "")
            texts[str(key)] = value
    else:
        for item in raw:
            key = item.get("id", item.get("doc_id"))
            texts[str(key)] = item["text"]
    return texts

def first_key(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError(str(names) + " not in " + str(list(d.keys())))

def parse_annotation(a):
    if isinstance(a, dict):
        s = first_key(a, ["start", "start_char", "start_offset", "begin"])
        e = first_key(a, ["end", "end_char", "end_offset"])
        lab = first_key(a, ["label", "type", "entity_label"])
        txt = a.get("text", a.get("entity_text", None))
        return int(s), int(e), lab, txt
    txt = a[3] if len(a) > 3 else None
    return int(a[0]), int(a[1]), a[2], txt

def load_gold_raw(path):
    raw = read_json(path)
    gold = {}
    if isinstance(raw, dict):
        items = list(raw.items())
    else:
        items = [(it.get("id", it.get("doc_id")), it) for it in raw]
    for key, value in items:
        if isinstance(value, dict):
            value = value.get("entities", value.get("annotations", value.get("spans", [])))
        gold[str(key)] = [parse_annotation(a) for a in value]
    return gold

raw_text = {}
raw_gold = {}
raw_text["fold1"] = load_texts(LOCAL_FILES["fold1-text.json"])
raw_text["fold2"] = load_texts(LOCAL_FILES["fold2-text.json"])
raw_text["fold3"] = load_texts(LOCAL_FILES["fold3-text.json"])
raw_gold["fold1"] = load_gold_raw(LOCAL_FILES["fold1-gold.json"])
raw_gold["fold2"] = load_gold_raw(LOCAL_FILES["fold2-gold.json"])

DEFAULT_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABELS = []
for line in open(LOCAL_FILES["labels.txt"], encoding="utf-8").read().splitlines():
    line = line.strip()
    if line:
        LABELS.append(line.replace(",", " ").split()[0])
if len(LABELS) != 12 or set(LABELS) != set(DEFAULT_LABELS):
    print("WARNING: labels.txt parsed as", LABELS, "-> using the 12 official labels")
    LABELS = DEFAULT_LABELS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

for fold in ["fold1", "fold2", "fold3"]:
    n_chars = sum(len(t) for t in raw_text[fold].values())
    print(f"{fold}: {len(raw_text[fold])} docs, {n_chars:,} chars")
    REPORT[fold + "_raw_docs"] = len(raw_text[fold])
    REPORT[fold + "_raw_chars"] = n_chars

# every gold doc id must exist in the text file and vice versa
for fold in ["fold1", "fold2"]:
    missing_text = [d for d in raw_gold[fold] if d not in raw_text[fold]]
    missing_gold = [d for d in raw_text[fold] if d not in raw_gold[fold]]
    print(fold, "gold ids without text:", len(missing_text), "| text ids without gold entry:", len(missing_gold))
    REPORT[fold + "_gold_without_text"] = len(missing_text)
    REPORT[fold + "_text_without_gold"] = len(missing_gold)

## 3. Gold end-offset convention

In [ ]:
excl = 0
incl = 0
with_text = 0
for fold in ["fold1", "fold2"]:
    for doc_id, anns in raw_gold[fold].items():
        if doc_id not in raw_text[fold]:
            continue
        text = raw_text[fold][doc_id]
        for s, e, lab, ent in anns:
            if ent is None:
                continue
            with_text += 1
            if text[s:e] == ent:
                excl += 1
            if text[s:e + 1] == ent:
                incl += 1
print(f"annotations with entity text: {with_text} | text[s:e]==ent: {excl} | text[s:e+1]==ent: {incl}")
if with_text == 0:
    print("WARNING: no entity text in gold; assuming exclusive end")
    END_INCLUSIVE = False
else:
    END_INCLUSIVE = incl > excl
print("end offset is", "INCLUSIVE" if END_INCLUSIVE else "EXCLUSIVE")
REPORT["end_inclusive"] = END_INCLUSIVE
REPORT["annotations_with_text"] = with_text

## 4. Character audit and length-preserving cleaning

In [ ]:
char_counter = Counter()
for fold in ["fold1", "fold2", "fold3"]:
    for text in raw_text[fold].values():
        char_counter.update(text)
print("distinct characters:", len(char_counter))

PUNCT_MAP = {"‘": "'", "’": "'", "“": '"', "”": '"', "–": "-", "—": "-", "−": "-"}

def map_char(c):
    # returns exactly ONE character, so text length never changes
    if c == "\n" or c == " ":
        return c
    if c in EXTRA_SPACE_CHARS:
        return " "
    category = unicodedata.category(c)
    if category in ("Zs", "Zl", "Zp"):
        return " "
    if category == "Cc":              # tab, \r, vertical tab, form feed, other control characters
        return " "
    if NORMALISE_PUNCT and c in PUNCT_MAP:
        return PUNCT_MAP[c]
    return c

# what is in the data, by unicode category (non-ascii letters are expected: Hindi etc.)
by_category = Counter()
for c, n in char_counter.items():
    by_category[unicodedata.category(c)] += n
print("characters by unicode category:", dict(by_category.most_common()))

suspicious = []
for c, n in char_counter.items():
    category = unicodedata.category(c)
    if c in "\n ":
        continue
    if category in ("Zs", "Zl", "Zp", "Cc", "Cf", "Cs", "Co", "Cn") or c == "�":
        suspicious.append((c, n, category))
suspicious.sort(key=lambda x: -x[1])
print("\nsuspicious characters (char, count, category):")
for c, n, category in suspicious[:30]:
    action = "-> space" if map_char(c) == " " else "left as is"
    print(f"  U+{ord(c):04X} {unicodedata.name(c, '?'):<32} {n:>9,} {category} {action}")
REPORT["suspicious_chars"] = [{"code": "U+%04X" % ord(c), "count": n, "category": cat, "action": "space" if map_char(c) == " " else "kept"} for c, n, cat in suspicious]

TABLE = {}
for c in char_counter:
    new = map_char(c)
    if new != c:
        assert len(new) == 1
        TABLE[ord(c)] = new
print("\nchars that will be replaced:", len(TABLE), "| total replacements:", sum(char_counter[chr(o)] for o in TABLE))

def clean_text(text):
    if not CLEAN_TEXT:
        return text
    new = text.translate(TABLE)
    assert len(new) == len(text)      # offsets stay valid
    return new

clean_text_by_fold = {}
for fold in ["fold1", "fold2", "fold3"]:
    clean_text_by_fold[fold] = {}
    changed_docs = 0
    for doc_id, text in raw_text[fold].items():
        new = clean_text(text)
        if new != text:
            changed_docs += 1
        clean_text_by_fold[fold][doc_id] = new
    print(fold, "docs changed by cleaning:", changed_docs, "/", len(raw_text[fold]))
    REPORT[fold + "_docs_changed_by_cleaning"] = changed_docs

# other text problems worth knowing about
for fold in ["fold1", "fold2", "fold3"]:
    texts = clean_text_by_fold[fold]
    empty = [d for d, t in texts.items() if len(t.strip()) == 0]
    short = [d for d, t in texts.items() if 0 < len(t.strip()) < 200]
    long_runs = sum(len(re.findall(r" {6,}", t)) for t in texts.values())
    many_newlines = sum(len(re.findall(r"\n{4,}", t)) for t in texts.values())
    print(f"{fold}: empty docs {len(empty)} | very short docs (<200 chars) {len(short)} | runs of 6+ spaces {long_runs} | runs of 4+ newlines {many_newlines}")
    REPORT[fold + "_empty_docs"] = len(empty)
    REPORT[fold + "_short_docs"] = len(short)
    REPORT[fold + "_space_runs"] = long_runs

## 5. Validate and repair gold spans

In [ ]:
def nearest_occurrence(text, ent, s, window):
    positions = []
    start = max(0, s - window)
    pos = text.find(ent, start)
    while pos != -1 and pos <= s + window:
        positions.append(pos)
        pos = text.find(ent, pos + 1)
    if not positions:
        return None
    positions.sort(key=lambda p: abs(p - s))
    if len(positions) > 1 and abs(positions[0] - s) == abs(positions[1] - s):
        return None                      # ambiguous
    return positions[0]

def validate_gold(fold):
    stats = Counter()
    clean_gold = {}
    for doc_id, anns in raw_gold[fold].items():
        text = raw_text[fold].get(doc_id)
        if text is None:
            stats["doc_without_text"] += len(anns)
            continue
        ctext = clean_text_by_fold[fold][doc_id]
        kept = []
        seen = set()
        for s, e, lab, ent in anns:
            stats["annotations"] += 1
            if lab not in LABEL2ID:
                stats["bad_label_dropped"] += 1
                continue
            e_excl = e + 1 if END_INCLUSIVE else e
            if s < 0 or e_excl > len(text) or s >= e_excl:
                stats["bad_range_dropped"] += 1
                continue
            if ent is not None and text[s:e_excl] != ent:
                stats["text_mismatch"] += 1
                pos = nearest_occurrence(text, ent, s, REPAIR_WINDOW) if len(ent) > 0 else None
                if pos is not None:
                    s = pos
                    e_excl = pos + len(ent)
                    stats["text_mismatch_repaired"] += 1
                else:
                    stats["text_mismatch_unrepaired"] += 1
            span_text = ctext[s:e_excl]
            if span_text != span_text.strip():
                stats["span_has_edge_whitespace"] += 1
            key = (s, e_excl, lab)
            if key in seen:
                stats["duplicate_span_dropped"] += 1
                continue
            seen.add(key)
            kept.append((s, e_excl, lab, span_text))
        kept.sort()
        # overlaps between different spans (nested / crossing entities)
        for i in range(len(kept)):
            for j in range(i + 1, len(kept)):
                if kept[j][0] >= kept[i][1]:
                    break
                stats["overlapping_pairs"] += 1
        clean_gold[doc_id] = kept
    return clean_gold, stats

clean_gold_by_fold = {}
for fold in ["fold1", "fold2"]:
    clean_gold_by_fold[fold], stats = validate_gold(fold)
    print(fold, dict(stats))
    REPORT[fold + "_gold_validation"] = dict(stats)
    no_ann = sum(1 for d in clean_text_by_fold[fold] if len(clean_gold_by_fold[fold].get(d, [])) == 0)
    print(f"  docs with no entities: {no_ann}/{len(clean_text_by_fold[fold])}")
    REPORT[fold + "_docs_without_entities"] = no_ann

## 6. Duplicates and leakage

In [ ]:
def text_hash(t):
    return hashlib.md5(t.strip().encode("utf-8")).hexdigest()

hashes = {}
for fold in ["fold1", "fold2", "fold3"]:
    hashes[fold] = {d: text_hash(t) for d, t in clean_text_by_fold[fold].items()}

dropped = {"fold1": [], "fold2": []}
seen_hash = {}
for fold in ["fold1", "fold2"]:
    for doc_id, h in hashes[fold].items():
        if h in seen_hash:
            dropped[fold].append((doc_id, "duplicate of " + seen_hash[h]))
        else:
            seen_hash[h] = fold + ":" + doc_id

train_hashes = set(hashes["fold1"].values()) | set(hashes["fold2"].values())
test_in_train = [d for d, h in hashes["fold3"].items() if h in train_hashes]
test_dups = len(hashes["fold3"]) - len(set(hashes["fold3"].values()))
f2_in_f1 = [d for d, h in hashes["fold2"].items() if h in set(hashes["fold1"].values())]
print("fold2 docs whose text is also in fold1 (validation leakage):", len(f2_in_f1))
print("duplicate docs inside fold1:", len(hashes["fold1"]) - len(set(hashes["fold1"].values())),
      "| inside fold2:", len(hashes["fold2"]) - len(set(hashes["fold2"].values())))
print("test docs whose text also appears in train:", len(test_in_train), "| duplicate docs inside test:", test_dups)
REPORT["fold2_docs_also_in_fold1"] = len(f2_in_f1)
REPORT["test_docs_also_in_train"] = len(test_in_train)
REPORT["test_internal_duplicates"] = test_dups
REPORT["dedup_dropped"] = {k: len(v) for k, v in dropped.items()}

final_text = {}
final_gold = {}
for fold in ["fold1", "fold2"]:
    drop_ids = set(d for d, why in dropped[fold]) if DEDUP_TRAIN else set()
    final_text[fold] = {d: t for d, t in clean_text_by_fold[fold].items() if d not in drop_ids}
    final_gold[fold] = {d: clean_gold_by_fold[fold].get(d, []) for d in final_text[fold]}
    print(fold, "docs kept:", len(final_text[fold]), "| dropped:", len(drop_ids))
final_text["fold3"] = clean_text_by_fold["fold3"]          # the test set is never filtered
print("fold3 docs kept:", len(final_text["fold3"]))

## 7. Tokenise, align, and prove the boundaries survive (oracle check)

In [ ]:
from transformers import AutoTokenizer

def find_tokenizer():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
             if "checkpoint-" not in p and "legal-gqa-final" not in p]
    if not found:
        raise FileNotFoundError("Attach your tokenizer dataset as an input.")
    with_meta = [d for d in found if os.path.exists(os.path.join(d, "trained_on.json"))]
    return (with_meta or found)[0]

TOK_PATH = find_tokenizer()
tokenizer = AutoTokenizer.from_pretrained(TOK_PATH)
tokenizer.model_max_length = 10**9
assert tokenizer.is_fast
EOS_ID = tokenizer.eos_token_id
print("tokenizer:", TOK_PATH, "| vocab", len(tokenizer), "| eos", EOS_ID)

def encode_doc(text):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc["input_ids"])
    starts = []
    ends = []
    real = []
    for s, e in enc["offset_mapping"]:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        starts.append(s)
        ends.append(e)
        real.append(e > s)
    if ADD_EOS_PREFIX and EOS_ID is not None:
        ids = [EOS_ID] + ids
        starts = [0] + starts
        ends = [0] + ends
        real = [False] + real
    doc = {}
    doc["ids"] = np.array(ids, dtype=np.int64)
    doc["starts"] = np.array(starts, dtype=np.int64)
    doc["ends"] = np.array(ends, dtype=np.int64)
    doc["real"] = np.array(real, dtype=bool)
    return doc

def make_tags(doc, spans):
    n = len(doc["ids"])
    tags = np.zeros(n, dtype=np.int64)
    taken = np.zeros(n, dtype=bool)
    for s, e, lab in sorted(spans, key=lambda x: -(x[1] - x[0])):
        first = int(np.searchsorted(doc["ends"], s, side="right"))
        last = int(np.searchsorted(doc["starts"], e, side="left"))
        idx = [i for i in range(first, last) if doc["real"][i] and not taken[i]]
        if not idx:
            continue
        li = LABEL2ID[lab]
        for k, i in enumerate(idx):
            tags[i] = 1 + 2 * li if k == 0 else 2 + 2 * li
            taken[i] = True
    tags[~doc["real"]] = -100
    return tags

def tags_to_spans(tags, starts, ends):
    spans = []
    cur = None
    cs = 0
    ce = 0
    for i in range(len(tags)):
        t = int(tags[i])
        if t == 0:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
                cur = None
            continue
        lab = (t - 1) // 2
        is_begin = (t - 1) % 2 == 0
        if is_begin or cur != lab:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
            cur = lab
            cs = starts[i]
        ce = ends[i]
    if cur is not None:
        spans.append((int(cs), int(ce), LABELS[cur]))
    return spans

def strict_scores(gold, pred):
    per_label = {}
    for lab in LABELS:
        tp = 0
        n_pred = 0
        n_gold = 0
        for doc_id in gold:
            g = set(x for x in gold[doc_id] if x[2] == lab)
            p = set(x for x in pred.get(doc_id, []) if x[2] == lab)
            tp += len(g & p)
            n_pred += len(p)
            n_gold += len(g)
        precision = tp / n_pred if n_pred else 0.0
        recall = tp / n_gold if n_gold else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[lab] = {"p": precision, "r": recall, "f1": f1, "gold": n_gold, "pred": n_pred}
    used = [l for l in LABELS if per_label[l]["gold"] > 0 or per_label[l]["pred"] > 0] or LABELS
    return float(np.mean([per_label[l]["f1"] for l in used])), per_label

def build_docs(texts, gold=None):
    docs = []
    for doc_id, text in texts.items():
        doc = encode_doc(text)
        doc["id"] = doc_id
        if gold is not None:
            doc["gold"] = [(s, e, lab) for s, e, lab, _ in gold.get(doc_id, [])]
            doc["tags"] = make_tags(doc, doc["gold"])
        docs.append(doc)
    return docs

def oracle(docs):
    gold = {}
    pred = {}
    for d in docs:
        idx = np.where(d["real"])[0]
        gold[d["id"]] = d["gold"]
        pred[d["id"]] = tags_to_spans(d["tags"][idx], d["starts"][idx], d["ends"][idx])
    return strict_scores(gold, pred)

t0 = time.time()
tok_docs = {}
tok_docs["fold1"] = build_docs(final_text["fold1"], final_gold["fold1"])
tok_docs["fold2"] = build_docs(final_text["fold2"], final_gold["fold2"])
tok_docs["fold3"] = build_docs(final_text["fold3"])
print(f"tokenised in {time.time() - t0:.0f}s")
for fold in ["fold1", "fold2", "fold3"]:
    for d in tok_docs[fold]:
        assert np.all(np.diff(d["starts"]) >= 0) and np.all(np.diff(d["ends"]) >= 0), "non-monotonic offsets in " + d["id"]

for fold in ["fold1", "fold2"]:
    f1, per_label = oracle(tok_docs[fold])
    print(f"ORACLE {fold}: macro-F1 {f1:.4f}")
    low = [(l, round(per_label[l]["f1"], 3), per_label[l]["gold"]) for l in LABELS if per_label[l]["gold"] > 0 and per_label[l]["f1"] < 0.99]
    if low:
        print("   labels below 0.99 (label, f1, n_gold):", low)
    REPORT[fold + "_oracle_f1"] = f1
    REPORT[fold + "_oracle_per_label"] = {l: per_label[l]["f1"] for l in LABELS}
    if f1 < 0.97:
        print("   !! low oracle: gold spans that do not start/end on token boundaries. Examples:")
        shown = 0
        for d in tok_docs[fold]:
            idx = np.where(d["real"])[0]
            got = set(tags_to_spans(d["tags"][idx], d["starts"][idx], d["ends"][idx]))
            for s, e, lab in d["gold"]:
                if (s, e, lab) not in got and shown < 8:
                    text = final_text[fold][d["id"]]
                    print("     ", lab, repr(text[max(0, s - 15):s]), "[[", repr(text[s:e]), "]]", repr(text[e:e + 15]))
                    shown += 1

## 8. Dataset statistics (for the report)

In [ ]:
def pct(arr, q):
    return float(np.percentile(arr, q)) if len(arr) else 0.0

stats_out = {}
for fold in ["fold1", "fold2", "fold3"]:
    docs = tok_docs[fold]
    n_tokens = np.array([len(d["ids"]) for d in docs])
    n_chars = np.array([len(final_text[fold][d["id"]]) for d in docs])
    n_windows = 0
    stride = MAX_LEN - OVERLAP
    for n in n_tokens:
        n_windows += 1 if n <= MAX_LEN else 1 + int(np.ceil((n - MAX_LEN) / stride))
    row = {"docs": len(docs), "tokens": int(n_tokens.sum()), "chars": int(n_chars.sum()),
           "chars_per_token": float(n_chars.sum() / max(1, n_tokens.sum())),
           "tokens_p50": pct(n_tokens, 50), "tokens_p90": pct(n_tokens, 90), "tokens_max": int(n_tokens.max()),
           "docs_longer_than_max_len": int((n_tokens > MAX_LEN).sum()), "windows": n_windows}
    stats_out[fold] = row
    print(fold, row)

print("\nentities per label:")
print(f"{'label':<10} {'fold1':>8} {'fold2':>8}")
label_counts = {}
for lab in LABELS:
    c1 = sum(1 for d in tok_docs["fold1"] for s, e, l in d["gold"] if l == lab)
    c2 = sum(1 for d in tok_docs["fold2"] for s, e, l in d["gold"] if l == lab)
    label_counts[lab] = [c1, c2]
    flag = "   <- rare" if c1 + c2 < 200 else ""
    print(f"{lab:<10} {c1:>8} {c2:>8}{flag}")
stats_out["label_counts"] = label_counts

tag_total = np.zeros(1 + 2 * len(LABELS))
for d in tok_docs["fold1"]:
    t = d["tags"]
    tag_total += np.bincount(t[t >= 0], minlength=len(tag_total))
print("\nfraction of real tokens tagged O in fold1: %.3f" % (tag_total[0] / tag_total.sum()))
stats_out["fold1_O_fraction"] = float(tag_total[0] / tag_total.sum())

lens = []
for d in tok_docs["fold1"]:
    t = d["tags"]
    run = 0
    for v in t:
        if v > 0 and (v - 1) % 2 == 0:
            if run:
                lens.append(run)
            run = 1
        elif v > 0:
            run += 1
        else:
            if run:
                lens.append(run)
            run = 0
print("entity length in tokens: median %.0f | p95 %.0f | max %d" % (pct(lens, 50), pct(lens, 95), max(lens) if lens else 0))
stats_out["entity_len_tokens"] = {"p50": pct(lens, 50), "p95": pct(lens, 95)}
REPORT["stats"] = stats_out

## 9. Save

In [ ]:
os.makedirs(os.path.join(OUT_DIR, "Train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "Test"), exist_ok=True)

def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

for fold in ["fold1", "fold2"]:
    write_json(final_text[fold], os.path.join(OUT_DIR, "Train", fold + "-text.json"))
    out_gold = {}
    for doc_id, spans in final_gold[fold].items():
        rows = []
        for s, e, lab, ent in spans:
            rows.append([s, e - 1 if END_INCLUSIVE else e, lab, ent])     # same end convention as the source file
        out_gold[doc_id] = rows
    write_json(out_gold, os.path.join(OUT_DIR, "Train", fold + "-gold.json"))
write_json(final_text["fold3"], os.path.join(OUT_DIR, "Test", "fold3-text.json"))
with open(os.path.join(OUT_DIR, "Train", "labels.txt"), "w") as f:
    f.write("\n".join(LABELS))

cache = {"labels": LABELS, "end_inclusive": END_INCLUSIVE, "add_eos_prefix": ADD_EOS_PREFIX,
         "tokenizer_path": TOK_PATH, "vocab_size": len(tokenizer), "eos_id": EOS_ID, "docs": tok_docs}
with open(os.path.join(OUT_DIR, "tokenized_ner.pkl"), "wb") as f:
    pickle.dump(cache, f)

REPORT["settings"] = {"CLEAN_TEXT": CLEAN_TEXT, "NORMALISE_PUNCT": NORMALISE_PUNCT, "DEDUP_TRAIN": DEDUP_TRAIN,
                      "ADD_EOS_PREFIX": ADD_EOS_PREFIX, "REPAIR_WINDOW": REPAIR_WINDOW}
REPORT["dropped_docs"] = {k: v for k, v in dropped.items()}
write_json(REPORT, os.path.join(OUT_DIR, "audit_report.json"))
for root, dirs, files in os.walk(OUT_DIR):
    for name in files:
        p = os.path.join(root, name)
        print(f"{os.path.relpath(p, OUT_DIR):<32} {os.path.getsize(p) / 1e6:8.2f} MB")

## 10. Re-load the saved files and verify

In [ ]:
check_text = {}
check_gold = {}
for fold in ["fold1", "fold2"]:
    check_text[fold] = load_texts(os.path.join(OUT_DIR, "Train", fold + "-text.json"))
    check_gold[fold] = load_gold_raw(os.path.join(OUT_DIR, "Train", fold + "-gold.json"))
check_text["fold3"] = load_texts(os.path.join(OUT_DIR, "Test", "fold3-text.json"))

problems = 0
# 1. test texts keep EXACTLY the original length, so predictions on them are valid for the original files too
for doc_id, text in raw_text["fold3"].items():
    if len(check_text["fold3"][doc_id]) != len(text):
        problems += 1
print("test docs with changed length:", problems, "(must be 0)")
assert problems == 0

# 2. every saved gold span reads back as its entity text under the saved end convention
bad = 0
total = 0
for fold in ["fold1", "fold2"]:
    for doc_id, anns in check_gold[fold].items():
        text = check_text[fold][doc_id]
        for s, e, lab, ent in anns:
            total += 1
            e_excl = e + 1 if END_INCLUSIVE else e
            if text[s:e_excl] != ent or lab not in LABEL2ID:
                bad += 1
print(f"saved gold spans that do not match their text: {bad}/{total} (must be 0)")
assert bad == 0

# 3. the cache is reproducible from the saved files
cache2 = pickle.load(open(os.path.join(OUT_DIR, "tokenized_ner.pkl"), "rb"))
for fold in ["fold1", "fold2", "fold3"]:
    for d_cache in cache2["docs"][fold][:20]:
        d_new = encode_doc(check_text[fold][d_cache["id"]])
        assert np.array_equal(d_new["ids"], d_cache["ids"]) and np.array_equal(d_new["ends"], d_cache["ends"])
print("tokenised cache matches a fresh tokenisation of the saved files")

print("\nSUMMARY")
print("  oracle fold1 %.4f | fold2 %.4f" % (REPORT["fold1_oracle_f1"], REPORT["fold2_oracle_f1"]))
print("  docs: fold1", len(final_text["fold1"]), "| fold2", len(final_text["fold2"]), "| test", len(final_text["fold3"]))
print("  gold fixes:", {k: v for k, v in REPORT["fold1_gold_validation"].items() if k not in ("annotations",)})

**Next:** Save a version of this notebook, then attach its *output* to the NER notebooks. They find `fold*-text.json` / `fold*-gold.json` / `labels.txt` automatically, so the cleaned files are drop-in.
The audit numbers (suspicious characters, gold repairs, duplicates, oracle, rare labels) are in `audit_report.json` for the report.